# parameter recovery

this is the same as the previous version but this time we only do param recovery on the non gray balloons!

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
os.environ["OMP_NUM_THREADS"] = "1"
import seaborn as sns 
from scipy.io import loadmat
import ast
from scipy.io import loadmat, savemat
import warnings
warnings.filterwarnings("ignore")
import statsmodels.api as sm


In [2]:
outputFolderName = r"param_recovery_4_param_recovery_v2"
inputFolderName = r"\\155.100.91.44\d\Data\Nill\BART_param_recovery\old_modeling\param_recovery_3_simulated_fields"

if not os.path.exists(outputFolderName):
    os.makedirs(outputFolderName)

In [3]:

matFiles = [f for f in os.listdir(inputFolderName) if f.endswith(".mat")]
nPatients = len(matFiles)
rows = []

for pt in range(nPatients):
# for pt in range(2):
    fileName = matFiles[pt]

    ptID = os.path.splitext(fileName)[0]
    ptID = ptID.replace("_TDdataParamRecovery", "")

    print(f"processing pt {pt+1}/{nPatients}: {ptID}")

    matFile = os.path.join(inputFolderName, fileName)
    mat = loadmat(matFile, struct_as_record=False, squeeze_me=True)

    TDdataParamRecovery = mat["TDdataParamRecovery"]

    alphas = np.asarray(TDdataParamRecovery.a, dtype=float)
    rewardSimulated = np.asarray(TDdataParamRecovery.rewardSimulated, dtype=float)

    result_raw = np.asarray(TDdataParamRecovery.resultSimulated, dtype=str)
    resultSimulated = np.array([1 if x == "popped" else 0 for x in result_raw], dtype=float)

    trial_types = np.asarray(TDdataParamRecovery.trial_type)

        
    # remove trial_type == 4 for all trial-wise data
    valid_mask = trial_types != 4
    rewardSimulated = rewardSimulated[valid_mask]
    resultSimulated = resultSimulated[valid_mask]
    trial_types = trial_types[valid_mask]

    nTrials = len(rewardSimulated)


    inverseTemperatureRSTD = np.full((len(alphas), len(alphas)), np.nan)

    for ap in range(len(alphas) - 1, -1, -1):
        for an in range(len(alphas) - 1, -1, -1):

            RewardPE = np.zeros(nTrials, dtype=float)
            expectedReward = np.zeros(nTrials, dtype=float)

            for t in range(1, nTrials):   # MATLAB t = 2:nTrials
                RewardPE[t] = rewardSimulated[t] - expectedReward[t - 1]

                if RewardPE[t] > 0:
                    expectedReward[t] = expectedReward[t - 1] + alphas[ap] * RewardPE[t - 1]
                elif RewardPE[t] < 0:
                    expectedReward[t] = expectedReward[t - 1] + alphas[an] * RewardPE[t - 1]
                else:
                    expectedReward[t] = expectedReward[t - 1]

            X = sm.add_constant(expectedReward)
            y = resultSimulated

            model = sm.GLM(
                y,
                X,
                family=sm.families.Binomial(link=sm.families.links.Logit())
            )
            result = model.fit()
            inverseTemperatureRSTD[ap, an] = result.params[1]

    # best alpha+ and alpha-
    best_idx = np.unravel_index(np.nanargmax(inverseTemperatureRSTD), inverseTemperatureRSTD.shape)
    bestAlphaPosIdx, bestAlphaNegIdx = best_idx

    bestAlphaPos = alphas[bestAlphaPosIdx]
    bestAlphaNeg = alphas[bestAlphaNegIdx]

    rows.append({
        "ptID": ptID,
        "fit_alpha_plus": TDdataParamRecovery.bestAlphaPos,
        "fit_alpha_minus": TDdataParamRecovery.bestAlphaNeg,
        "sim_alpha_plus": bestAlphaPos,
        "sim_alpha_minus": bestAlphaNeg
    })

# save to csv
df_alpha_compare = pd.DataFrame(rows)

output_csv = os.path.join(outputFolderName, "alpha_comparison.csv")
df_alpha_compare.to_csv(output_csv, index=False)

print(f"saved: {output_csv}")

processing pt 1/71: 201810


IndexError: boolean index did not match indexed array along dimension 0; dimension is 213 but corresponding boolean dimension is 214

# debug

In [ ]:
fields = [f for f in dir(TDdataParamRecovery) if not f.startswith('_')]
print(fields)

['Reward', 'a', 'bestAlphaNeg', 'bestAlphaPos', 'inflate_time', 'inverseTemperatureRSTD', 'is_control', 'nTrials', 'points', 'pointsMinusReward', 'result', 'resultSimulated', 'rewardSimulated', 'rstdRPE', 'rstdV', 'score', 'trial_type']
